In [1]:
import os
import sys
import torch
import numpy as np


from PIL import Image
from torchvision import transforms
from torchvision.transforms.functional import pil_to_tensor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights


class ResNetMoleClassifier(nn.Module):
    def __init__(self, num_classes=4, pretrained=True):
        super(ResNetMoleClassifier, self).__init__()

        self.resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)

        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.resnet(x)


def create_resnet_model(num_classes=4, pretrained=True):
    return ResNetMoleClassifier(num_classes, pretrained)

In [14]:
img_path = "../assets/P_128_crop_0_dermal.jpg"

if os.path.exists(img_path):
    print(f"Image found at: {img_path}")
else:
    print(f"Image not found at: {img_path}")

Image found at: ../assets/P_128_crop_0_dermal.jpg


In [15]:
resnet_model = create_resnet_model(num_classes=4, pretrained=True).to(DEVICE)

OUR_CKPT = "../Experiment/resnet/SemanticMix+SSI/best_model.pth"

if os.path.exists(OUR_CKPT):
    print(f"Checkpoint found at: {OUR_CKPT}")
else:
    print(f"Checkpoint not found at: {OUR_CKPT}")

if os.path.isfile(OUR_CKPT):
    print("loading ckpt...")
    resnet_model.load_state_dict(torch.load(OUR_CKPT)['model_state_dict'])
    resnet_model.eval()
    print("load successfully...")

Checkpoint found at: ../Experiment/resnet/SemanticMix+SSI/best_model.pth
loading ckpt...
load successfully...


In [16]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


def preprocess_image(image_path):
    image_pil = Image.open(img_path).convert('RGB')
    tensor = transform(image_pil)
    return tensor.unsqueeze(0).cuda()

image_tensor = preprocess_image(img_path)

with torch.no_grad():
    outputs = resnet_model(image_tensor)
    probabilities = torch.softmax(outputs, dim=1)
    predicted_class = torch.argmax(probabilities, dim=1).item()

print(f"\n=== Prediction Results ===")
print(f"Predicted Class ID: {predicted_class}")
print(f"Confidence Scores:")
for idx in range(len(probabilities[0])):
    print(f"  Class {idx}: {probabilities[0][idx].item():.4f}")

CLASS_NAMES = ["Compound", "Junctional", 
               "Dermal", "Seborrheic"]

pred_label = CLASS_NAMES[predicted_class]
top_confidence = probabilities[0][predicted_class].item() * 100

print(f"\n Diagnosis: {pred_label}")
print(f" Confidence: {top_confidence:.2f}%")


=== Prediction Results ===
Predicted Class ID: 2
Confidence Scores:
  Class 0: 0.0053
  Class 1: 0.0057
  Class 2: 0.9875
  Class 3: 0.0015

 Diagnosis: Dermal
 Confidence: 98.75%
